# 01 · Download + index

`download` expands every playlist / channel / video link in `links.txt` into **unique** videos, then fetches the best audio stream as **FLAC, 24 kHz, mono** (one video at a time, IPv4, yt-dlp download archive: resumable, no cookies needed).
Playlist videos go to `<download.dir>/<playlist title>/`, single videos to `<download.dir>/`.

`index` lists the episodes on disk (any folder of `<title> [<video id>].flac` files works, e.g. episodes downloaded earlier) with their durations. Every later step reads this index.

In [ ]:
import os, sys, json
from pathlib import Path
REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO)); os.chdir(REPO)
if Path("/workspace").exists():
    os.environ.setdefault("HF_HOME", "/workspace/hf_cache")
import numpy as np, pandas as pd
import plotly.express as px
from egyspeech import nb
cfg, lay = nb.cfg_and_layout()          # configs/config.yaml
print("work dir:", lay.root)

In [ ]:
nb.run("download")          # add "--limit", "5" to try a few first

In [ ]:
nb.run("index")

In [ ]:
videos = pd.DataFrame(nb.read_jsonl(lay.videos))
print(len(videos), "episodes,", round(videos["duration"].sum() / 3600, 1), "hours")
display(videos.assign(playlist=videos["playlist"].fillna("(no playlist)")).groupby("playlist")["duration"]
        .agg(["count", lambda s: round(s.sum() / 3600, 1)]).rename(columns={"<lambda_0>": "hours"})
        .sort_values("hours", ascending=False).head(30))
px.histogram(videos, x=videos["duration"] / 60, nbins=50, title="Episode length (min)")

In [ ]:
fails = pd.DataFrame(nb.read_jsonl(lay.failures("download")))
print(len(fails), "download failures in the last run"); display(fails.head(20))

In [ ]:
v = videos.iloc[0]
nb.play(v["path"], f"**{v['title']}** (first 30 s)", max_sec=30)